## Structured Output in LangChain
Normally LLM returns plain text. But sometimes we need the output in a fixed format like JSON or a Pydantic object.

Use `.with_structured_output()` to force the model to follow a specific schema.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model='openai/gpt-oss-20b')

### Pydantic schema for structured output
Define a Pydantic `BaseModel` with field types and descriptions. Pass it to `.with_structured_output()`.

Adding `description` to each field is important — it tells the model what value to fill in.

In [ ]:
from pydantic import BaseModel, Field

class MovieInfo(BaseModel):
    title: str = Field(description='Movie ka naam')
    director: str = Field(description='Director ka naam')
    year: int = Field(description='Release year, jaise 2020')
    rating: float = Field(description='IMDB rating out of 10')
    genre: str = Field(description='Movie ka genre jaise Action, Drama etc')

# model ko structured output dene ka instruction
structured_model = model.with_structured_output(MovieInfo)

result = structured_model.invoke('Tell me about the movie Inception')
print(result)
print(type(result))

### Converting result to dictionary
The result is a Pydantic object. Use `.model_dump()` to convert it to a dict.

In [ ]:
# Pydantic object ko dict mein convert karna
result_dict = result.model_dump()
print(result_dict)
print()

# individual fields access
print(f'Movie: {result.title}')
print(f'Director: {result.director}')
print(f'Year: {result.year}')
print(f'Rating: {result.rating}')

### Extracting multiple items — List fields
Use `List` type in a field when you need the model to return multiple values.

In [ ]:
from typing import List

class BookRecommendation(BaseModel):
    topic: str = Field(description='Kis topic ke liye books recommend ki hain')
    books: List[str] = Field(description='Recommended books ki list')
    difficulty: str = Field(description='Beginner, Intermediate ya Advanced')

book_model = model.with_structured_output(BookRecommendation)

resp = book_model.invoke('Python seekhne ke liye 3 books recommend karo beginners ke liye')
print(f'Topic: {resp.topic}')
print(f'Difficulty: {resp.difficulty}')
for i, book in enumerate(resp.books, 1):
    print(f'  {i}. {book}')

### Nested schema
For complex data, use nested Pydantic models. For example, a person object containing an address object.

In [ ]:
class Address(BaseModel):
    city: str = Field(description='City ka naam')
    state: str = Field(description='State ka naam')
    country: str = Field(description='Country ka naam')

class PersonInfo(BaseModel):
    name: str = Field(description='Person ka naam')
    age: int = Field(description='Person ki age')
    profession: str = Field(description='Kya kaam karta hai')
    address: Address = Field(description='Person ka address')

person_model = model.with_structured_output(PersonInfo)

resp = person_model.invoke('Sundar Pichai ke baare mein batao')
print(f'Name: {resp.name}')
print(f'Age: {resp.age}')
print(f'Profession: {resp.profession}')
print(f'City: {resp.address.city}')
print(f'Country: {resp.address.country}')

### Fixed choices using Literal
Use `Literal` to restrict a field to specific allowed values only.

In [ ]:
from typing import Literal

class SentimentResult(BaseModel):
    text: str = Field(description='Original text jo analyze kiya')
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(description='Sentiment of the text')
    confidence: float = Field(description='Confidence score 0 to 1')

sentiment_model = model.with_structured_output(SentimentResult)

texts = [
    'Ye movie bahut zabardast thi! Maza aa gaya',
    'Khaana bilkul bekar tha, dobara nahi jaunga',
    'Theek hai, kuch khaas nahi tha'
]

for t in texts:
    r = sentiment_model.invoke(t)
    print(f'Text: {r.text[:40]}... | Sentiment: {r.sentiment} | Confidence: {r.confidence}')

### Optional fields
Use `Optional` for fields that may or may not be present in the output.

In [ ]:
from typing import Optional

class ProductReview(BaseModel):
    product_name: str = Field(description='Product ka naam')
    rating: int = Field(description='Rating 1 se 5 tak')
    pros: List[str] = Field(description='Achhi baatein')
    cons: List[str] = Field(description='Buri baatein')
    suggestion: Optional[str] = Field(default=None, description='Koi suggestion ho toh')

review_model = model.with_structured_output(ProductReview)

resp = review_model.invoke('iPhone 15 ka review do — honest wala')
print(f'Product: {resp.product_name}')
print(f'Rating: {resp.rating}/5')
print(f'Pros: {resp.pros}')
print(f'Cons: {resp.cons}')
if resp.suggestion:
    print(f'Suggestion: {resp.suggestion}')

### Structured output with a chain
Combine prompt template with structured model to create a reusable chain.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

class CityInfo(BaseModel):
    name: str = Field(description='City ka naam')
    population: str = Field(description='Approx population')
    famous_for: List[str] = Field(description='Kis cheez ke liye famous hai')
    best_time_to_visit: str = Field(description='Ghumne ka best time')

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Tum ek travel expert ho. Accurate info do.'),
    ('human', '{city} ke baare mein batao')
])

chain = prompt | model.with_structured_output(CityInfo)

result = chain.invoke({'city': 'Jaipur'})
print(f'City: {result.name}')
print(f'Population: {result.population}')
print(f'Famous for: {", ".join(result.famous_for)}')
print(f'Best time: {result.best_time_to_visit}')

In [ ]:
# same chain, different city
result2 = chain.invoke({'city': 'Varanasi'})
print(f'City: {result2.name}')
print(f'Population: {result2.population}')
print(f'Famous for: {", ".join(result2.famous_for)}')
print(f'Best time: {result2.best_time_to_visit}')

### TypedDict alternative
If you don't want to use Pydantic, `TypedDict` also works. But Pydantic is better because it has built-in validation.

In [ ]:
from typing import TypedDict, Annotated

class JokeOutput(TypedDict):
    setup: Annotated[str, 'joke ka setup part']
    punchline: Annotated[str, 'joke ka punchline']
    category: Annotated[str, 'joke ka category jaise tech, dad joke etc']

joke_model = model.with_structured_output(JokeOutput)

joke = joke_model.invoke('ek programming wala joke sunao')
print(f'Setup: {joke["setup"]}')
print(f'Punchline: {joke["punchline"]}')
print(f'Category: {joke["category"]}')

### Error handling
Sometimes the model doesn't follow the schema properly. Use try-except to handle these cases.

In [ ]:
# error handling example
try:
    result = structured_model.invoke('Tell me about the movie 3 Idiots')
    print(f'Movie: {result.title}')
    print(f'Director: {result.director}')
    print(f'Year: {result.year}')
    print(f'Rating: {result.rating}')
    print(f'Genre: {result.genre}')
except Exception as e:
    print(f'Error aaya: {e}')
    print('Retry karo ya model change karo')